In [0]:
#Load Data
from pyspark.sql.functions import *

telemetry = spark.table(
    "workspace.predictive_maintenance.silver_telemetry"
)

failures = spark.table(
    "workspace.predictive_maintenance.silver_failures"
)

In [0]:
#Calculate Sensor Statistics
telemetry.select(
    avg("volt").alias("avg_volt"),
    stddev("volt").alias("std_volt"),
    avg("pressure").alias("avg_pressure"),
    stddev("pressure").alias("std_pressure"),
    avg("vibration").alias("avg_vibration"),
    stddev("vibration").alias("std_vibration"),
    avg("rotate").alias("avg_rotate"),
    stddev("rotate").alias("std_rotate")
).show()

+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+
|          avg_volt|          std_volt|     avg_pressure|      std_pressure|     avg_vibration|     std_vibration|       avg_rotate|        std_rotate|
+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+
|170.77773639409244|15.509114206502083|100.8586679910788|11.048679392647113|40.385006761144794|5.3703605617183925|446.6051188962638|52.673886035632655|
+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+



In [0]:
#Create Anomaly Flags
from pyspark.sql.functions import col, when, abs

anomalies = telemetry.withColumn(
    "volt_anomaly",
    when(
        abs(col("volt") - 170.7777) > (3 * 15.5091),
        1
    ).otherwise(0)
).withColumn(
    "pressure_anomaly",
    when(
        abs(col("pressure") - 100.8587) > (3 * 11.0487),
        1
    ).otherwise(0)
).withColumn(
    "vibration_anomaly",
    when(
        abs(col("vibration") - 40.3850) > (3 * 5.3704),
        1
    ).otherwise(0)
).withColumn(
    "rotate_anomaly",
    when(
        abs(col("rotate") - 446.6051) > (3 * 52.6739),
        1
    ).otherwise(0)
)

In [0]:
#Total Anomaly Score
anomalies = anomalies.withColumn(
    "anomaly_score",
    col("volt_anomaly")
    + col("pressure_anomaly")
    + col("vibration_anomaly")
    + col("rotate_anomaly")
)

display(
    anomalies.select(
        "machineID",
        "datetime",
        "anomaly_score"
    )
)

machineID,datetime,anomaly_score
1,2015-01-03T17:00:00.000Z,0
1,2015-01-06T14:00:00.000Z,0
1,2015-01-11T10:00:00.000Z,0
1,2015-01-13T18:00:00.000Z,0
1,2015-01-17T06:00:00.000Z,0
1,2015-01-19T07:00:00.000Z,0
1,2015-01-21T06:00:00.000Z,0
1,2015-01-23T08:00:00.000Z,0
1,2015-01-23T10:00:00.000Z,0
1,2015-01-23T20:00:00.000Z,0


In [0]:
#High-Risk Records
high_risk = anomalies.filter(
    col("anomaly_score") >= 2
)

display(high_risk)

datetime,machineID,volt,rotate,pressure,vibration,volt_anomaly,pressure_anomaly,vibration_anomaly,rotate_anomaly,anomaly_score
2015-12-29T09:00:00.000Z,15,152.905275351592,468.025516650634,143.532060377667,60.7617301753808,0,1,1,0,2
2015-08-04T15:00:00.000Z,21,189.47745623915,279.533937955216,80.979266084915,61.5811778830454,0,0,1,1,2
2015-07-02T13:00:00.000Z,29,156.43874246021,325.630161116758,144.468885036647,58.5445539959897,0,1,1,0,2
2015-01-05T13:00:00.000Z,62,191.022708920624,285.929409211394,137.160059972265,42.7142120119763,0,1,0,1,2
2015-07-08T11:00:00.000Z,80,185.775027308154,499.995973301946,140.483596487511,59.5489315797444,0,1,1,0,2
2015-12-24T15:00:00.000Z,33,166.944474813939,237.313344540743,136.126706703115,46.8630495325653,0,1,0,1,2
2015-09-29T17:00:00.000Z,37,118.863580294557,407.734494224773,109.104506907548,57.1192887313569,1,0,1,0,2
2015-12-29T08:00:00.000Z,15,157.126737550984,505.180429464823,137.299899617632,65.9304404528114,0,1,1,0,2
2015-03-12T12:00:00.000Z,36,227.161824248931,525.395527660814,101.760488130658,57.8315126480084,1,0,1,0,2
2015-03-10T21:00:00.000Z,77,141.851546971015,282.279178157873,140.502882764269,39.9214912649135,0,1,0,1,2


In [0]:
#Count Anomalies
print(
    "Total High Risk Records:",
    high_risk.count()
)

Total High Risk Records: 148


In [0]:
display(
    high_risk.groupBy("machineID").count()
)

machineID,count
15,6
21,3
29,1
62,1
80,2
33,2
37,3
36,2
77,1
45,3


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    high_risk.groupBy("anomaly_score").count()
)

anomaly_score,count
2,148


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    high_risk.select(
        "datetime",
        "anomaly_score"
    )
)

datetime,anomaly_score
2015-12-29T09:00:00.000Z,2
2015-08-04T15:00:00.000Z,2
2015-07-02T13:00:00.000Z,2
2015-01-05T13:00:00.000Z,2
2015-07-08T11:00:00.000Z,2
2015-12-24T15:00:00.000Z,2
2015-09-29T17:00:00.000Z,2
2015-12-29T08:00:00.000Z,2
2015-03-12T12:00:00.000Z,2
2015-03-10T21:00:00.000Z,2


Databricks visualization. Run in Databricks to view.